In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create output directory in Google Drive
import os
PROJECT_DIR = '/content/drive/MyDrive/Video_Translation_Project/Stage_1'
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"[INFO] Output directory ready at: {PROJECT_DIR}")

# Install dependencies
!pip install -q demucs noisereduce librosa soundfile
print("[SUCCESS] Libraries installed!")

Mounted at /content/drive
[INFO] Output directory ready at: /content/drive/MyDrive/Video_Translation_Project/Stage_1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 4.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.1/87.1 kB 6.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 5.1 MB/s eta 0:00:00
[SUCCESS] Libraries installed!


In [ ]:
import subprocess
import librosa
import noisereduce as nr
import soundfile as sf
import shutil
import os

# Define the input video name (must be uploaded to Colab's /content/ folder)
VIDEO_NAME = "Test.mp4"

# Local processing paths
input_video = f"/content/{VIDEO_NAME}"
raw_audio = "/content/raw_audio.wav"

if not os.path.exists(input_video):
    print(f"[ERROR] Video '{VIDEO_NAME}' not found in /content/. Please upload it.")
else:
    print("[INFO] Starting audio extraction and separation...")

    # 1. Create silent video and save directly to Drive
    print("[1/5] Creating silent video...")
    silent_vid_path = os.path.join(PROJECT_DIR, f"Silent_{VIDEO_NAME}")
    subprocess.run(['ffmpeg', '-y', '-i', input_video, '-c:v', 'copy', '-an', silent_vid_path],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # 2. Extract raw audio (16kHz, mono for best AI model compatibility)
    print("[2/5] Extracting raw audio...")
    subprocess.run(['ffmpeg', '-y', '-i', input_video, '-vn', '-acodec', 'pcm_s16le', '-ac', '1', '-ar', '16000', raw_audio],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # 3. Separate vocals from background noise using Demucs
    print("[3/5] Separating vocals and background noise (Demucs)...")
    subprocess.run(['python3', '-m', 'demucs.separate', '-n', 'htdemucs', '--two-stems', 'vocals', raw_audio],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    # Demucs default output paths
    demucs_vocals = "/content/separated/htdemucs/raw_audio/vocals.wav"
    demucs_noise = "/content/separated/htdemucs/raw_audio/no_vocals.wav"

    # 4. Apply noise reduction to vocals and save to Drive
    print("[4/5] Applying noise reduction to vocals...")
    audio_data, sr = librosa.load(demucs_vocals, sr=None)
    clean_audio_data = nr.reduce_noise(y=audio_data, sr=sr)

    clean_vocals_path = os.path.join(PROJECT_DIR, "Clean_Vocals.wav")
    sf.write(clean_vocals_path, clean_audio_data, sr)

    # 5. Save background noise to Drive
    print("[5/5] Saving background noise to Drive...")
    background_noise_path = os.path.join(PROJECT_DIR, "Background_Noise.wav")
    shutil.copy(demucs_noise, background_noise_path)

    print("\n[SUCCESS] Stage 1 Completed!")
    print("-" * 40)
    print(f"Silent Video : {silent_vid_path}")
    print(f"Clean Vocals : {clean_vocals_path}")
    print(f"Background   : {background_noise_path}")

[INFO] Starting audio extraction and separation...
[1/5] Creating silent video...
[2/5] Extracting raw audio...
[3/5] Separating vocals and background noise (Demucs)...
[4/5] Applying noise reduction to vocals...
[5/5] Saving background noise to Drive...

[SUCCESS] Stage 1 Completed!
----------------------------------------
Silent Video : /content/drive/MyDrive/Video_Translation_Project/Stage_1/Silent_Test.mp4
Clean Vocals : /content/drive/MyDrive/Video_Translation_Project/Stage_1/Clean_Vocals.wav
Background   : /content/drive/MyDrive/Video_Translation_Project/Stage_1/Background_Noise.wav
